# Greedy based agents

> Agents utelizing the Greedy approach for Dynamic pricing and learning problems

In [ ]:
#| default_exp agents.dynamic_pricing.greedy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction

In [ ]:
#| export
class _OLSIncremental:
    """
    Maintains   M_t  = λ I + Σ z_s z_sᵀ
           and   θ̂_t = M_t⁻¹ q_t      with q_t = Σ z_s D_s .
    O(d²) memory, O(d²) time per step.
    """
    def __init__(self, d, lam=1e-6):          # tiny λ keeps M invertible
        self.lam     = lam
        self.M_inv   = np.eye(d) / lam        # (λI)⁻¹
        self.q       = np.zeros(d)

    def update(self, z, D):
        z   = z.ravel().astype(float)
        D   = float(D)

        Mz      = self.M_inv @ z
        denom   = 1.0 + z @ Mz
        self.M_inv -= np.outer(Mz, Mz) / denom     # Sherman–Morrison rank-1
        self.q     += D * z

    @property
    def theta_hat(self):
        return self.M_inv @ self.q


In [ ]:
#| export
class GreedyPolicy:
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors=None,
                 actionprocessors=None,
                 agent_name=None,
                 ex_prices=None,
                 alpha=None,
                 beta=None,
                 price_function=None,
                 g=None):

        d_feat = environment_info.observation_space['features'].shape[0]
        if alpha is None:
            alpha = np.zeros(d_feat)
            beta  = np.zeros(d_feat)
        if isinstance(ex_prices, list):
            ex_prices = np.array(ex_prices)
        assert ex_prices.shape[0] >= 2

        # --- store -----------------------------------------------------------------
        self.environment_info = environment_info
        self.ex_prices        = ex_prices
        self.alpha            = alpha
        self.beta             = beta
        self.price_function   = price_function
        self.g                = g
        self.t                = 0
        self.mode             = "train"

        # incremental OLS on 2d parameters
        self._d_param   = 2 * d_feat
        self.estimator  = _OLSIncremental(self._d_param, lam=1e-6)

        # processors ----------------------------------------------------------------
        self.actionprocessors = actionprocessors or []
        self.actionprocessors.append(
            ClipAction(environment_info.action_space.low,
                       environment_info.action_space.high)
        )

    # --------------------------------------------------------------------- draw ---
    def draw_action(self, observation):
        if self.t < self.ex_prices.shape[0]:
            price = self.ex_prices[self.t]
        else:
            x     = observation['features']
            price = self.price_function(x, self.alpha, self.beta)

        for proc in self.actionprocessors:
            price = proc(price)
        return np.array(price, dtype=np.float32)

    # ----------------------------------------------------------------------- fit --
    def fit(self, X, Y, action):
        assert self.mode == "train"

        z = np.concatenate([X, X * action])      # length 2d
        self.estimator.update(z, Y)              # incremental OLS
        theta = self.estimator.theta_hat
        d     = theta.size // 2
        self.alpha, self.beta = theta[:d], theta[d:]

        self.t += 1

    # -------------------------------------------------------------- misc helpers --
    def update_task(self, env):
        self.environment_info = env.mdp_info
        self.t        = 0
        self._d_param = 2 * env.mdp_info.observation_space['features'].shape[0]
        self.estimator = _OLSIncremental(self._d_param, lam=1e-6)
        self.actionprocessors[-1] = ClipAction(
            env.mdp_info.action_space.low,
            env.mdp_info.action_space.high,
        )

    def reset(self):
        """Reset internal counters between episodes."""
        self.t = 0
        self.estimator = _OLSIncremental(self._d_param, lam=1e-6)


In [ ]:
#| export
class GreedyCoreAgent(Agent):

    """
    Base class for greedy bandit agents.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = GreedyPolicy(environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, ex_prices=ex_prices, alpha=alpha, beta=beta, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]["features"]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
    
    def update_task(self, env):
        self.policy.update_task(env)
        

In [ ]:
#| export
class GreedyAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for GreedyCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = GreedyCoreAgent(environment_info = environment_info,
                                     obsprocessors = obsprocessors, 
                                     actionprocessors = actionprocessors, 
                                     agent_name = agent_name, 
                                     ex_prices = ex_prices, 
                                     alpha = alpha, 
                                     beta = beta, 
                                     price_function = price_function, 
                                     g = g)
        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)
        